In [6]:
import python_obfuscator
from python_obfuscator.techniques import add_random_variables, one_liner, variable_renamer
import os
import shutil

In [5]:
PROJECT_EULER_PATH = "/home/danielrezende/datasets/PythonTheAlgorithms/project_euler"

# within each subfolder of PROJECT_EULER_PATH, pick the sol1.py, rename the file to subfolder_name.py, and copy to ./dataset/original
destination_path = "./dataset/original"
os.makedirs(destination_path, exist_ok=True)

for subfolder in os.listdir(PROJECT_EULER_PATH):
    subfolder_path = os.path.join(PROJECT_EULER_PATH, subfolder)
    if os.path.isdir(subfolder_path):
        source_file = os.path.join(subfolder_path, "sol1.py")
        if os.path.exists(source_file):
            destination_file = os.path.join(destination_path, f"{subfolder}.py")
            shutil.copy(source_file, destination_file)

In [9]:
# now obfuscate the code using python_obfuscator
def obfuscate(program: str, strategy: int):
    """
    Obfuscates a Python program.

    - program: the program to obfuscate, represented as a string
    - strategy: the obfuscation strategy to use (integer number from 0 to 7)
    """
    obfuscator = python_obfuscator.obfuscator()
    obfuscation_techniques = [
        [],  # 0 0 0
        [variable_renamer],
        [one_liner],
        [one_liner, variable_renamer],
        [add_random_variables],
        [add_random_variables, variable_renamer],
        [add_random_variables, one_liner],
        # [add_random_variables, one_liner, variable_renamer] | This won't do any obfuscation at all, and thus will be discarded
    ]
    obfuscated = obfuscator.obfuscate(
        program, remove_techniques=obfuscation_techniques[strategy]
    )
    if not isinstance(obfuscated, str):
        raise TypeError
    return obfuscated


# Generating all obfuscations for a given problem
# The strategies that passed the test in experiments.ipynb are deeemed safe,
# but we'll still have to double check the results anyway
SAFE_STRATEGIES = [2, 3, 6]


def generate_obfuscations(program: str)->list[str]:
    return [obfuscate(program, strategy) for strategy in SAFE_STRATEGIES]

In [14]:
# get the path of each file in ./dataset/original
original_files = [os.path.join(destination_path, file) for file in os.listdir(destination_path) if file.endswith('.py')]

# read each file, generate obfuscations, rename the new file to be {original_file.stem}_strat{i}.py, then write into ./dataset/obfuscated
obfuscated_path = "./dataset/obfuscated"
os.makedirs(obfuscated_path, exist_ok=True)

for original_file in original_files:
    with open(original_file, 'r') as f:
        program = f.read()
    obfuscations = generate_obfuscations(program)
    for i, obfuscated_program in enumerate(obfuscations):
        obfuscated_file = os.path.join(obfuscated_path, f"{os.path.splitext(os.path.basename(original_file))[0]}_strat_{i}.py")
        with open(obfuscated_file, 'w') as f:
            f.write(obfuscated_program)

In [3]:
import pandas as pd

file_path = (
    "../dolos-report-20250121T123750528Z-problem001py--problem001minifiedpy/pairs.csv"
)
data = pd.read_csv(file_path)
data.head()

,id,leftFileId,leftFilePath,rightFileId,rightFilePath,similarity,totalOverlap,longestFragment,leftCovered,rightCovered
0,24977,99,dataset/python_minifier/problem_054.py,102,dataset/python_minifier/problem_054_minified.py,0.767092,662,50,334,328
1,35534,234,dataset/python_minifier/problem_551.py,237,dataset/python_minifier/problem_551_minified.py,0.878906,450,112,225,225
2,32861,175,dataset/python_minifier/problem_101.py,172,dataset/python_minifier/problem_101_minified.py,0.623404,293,82,146,147
3,31734,154,dataset/python_minifier/problem_089.py,160,dataset/python_minifier/problem_089_minified.py,0.869565,280,79,140,140
4,35156,217,dataset/python_minifier/problem_180.py,218,dataset/python_minifier/problem_180_minified.py,0.720000,270,47,134,136


In [4]:
data['problem1'] = data['leftFilePath'].str.extract(r'problem_(\d+)', expand=False)
data.head()

,id,leftFileId,leftFilePath,rightFileId,rightFilePath,similarity,totalOverlap,longestFragment,leftCovered,rightCovered,problem1
0,24977,99,dataset/python_minifier/problem_054.py,102,dataset/python_minifier/problem_054_minified.py,0.767092,662,50,334,328,054
1,35534,234,dataset/python_minifier/problem_551.py,237,dataset/python_minifier/problem_551_minified.py,0.878906,450,112,225,225,551
2,32861,175,dataset/python_minifier/problem_101.py,172,dataset/python_minifier/problem_101_minified.py,0.623404,293,82,146,147,101
3,31734,154,dataset/python_minifier/problem_089.py,160,dataset/python_minifier/problem_089_minified.py,0.869565,280,79,140,140,089
4,35156,217,dataset/python_minifier/problem_180.py,218,dataset/python_minifier/problem_180_minified.py,0.720000,270,47,134,136,180


In [6]:
data['problem2'] = data['rightFilePath'].str.extract(r'problem_(\d+)', expand=False)
data.head()

,id,leftFileId,leftFilePath,rightFileId,rightFilePath,similarity,totalOverlap,longestFragment,leftCovered,rightCovered,problem1,problem2
0,24977,99,dataset/python_minifier/problem_054.py,102,dataset/python_minifier/problem_054_minified.py,0.767092,662,50,334,328,054,054
1,35534,234,dataset/python_minifier/problem_551.py,237,dataset/python_minifier/problem_551_minified.py,0.878906,450,112,225,225,551,551
2,32861,175,dataset/python_minifier/problem_101.py,172,dataset/python_minifier/problem_101_minified.py,0.623404,293,82,146,147,101,101
3,31734,154,dataset/python_minifier/problem_089.py,160,dataset/python_minifier/problem_089_minified.py,0.869565,280,79,140,140,089,089
4,35156,217,dataset/python_minifier/problem_180.py,218,dataset/python_minifier/problem_180_minified.py,0.720000,270,47,134,136,180,180


In [7]:
data['ground_truth'] = data['problem1'] == data['problem2']
data.head()

,id,leftFileId,leftFilePath,rightFileId,rightFilePath,similarity,totalOverlap,longestFragment,leftCovered,rightCovered,problem1,problem2,ground_truth
0,24977,99,dataset/python_minifier/problem_054.py,102,dataset/python_minifier/problem_054_minified.py,0.767092,662,50,334,328,054,054,True
1,35534,234,dataset/python_minifier/problem_551.py,237,dataset/python_minifier/problem_551_minified.py,0.878906,450,112,225,225,551,551,True
2,32861,175,dataset/python_minifier/problem_101.py,172,dataset/python_minifier/problem_101_minified.py,0.623404,293,82,146,147,101,101,True
3,31734,154,dataset/python_minifier/problem_089.py,160,dataset/python_minifier/problem_089_minified.py,0.869565,280,79,140,140,089,089,True
4,35156,217,dataset/python_minifier/problem_180.py,218,dataset/python_minifier/problem_180_minified.py,0.720000,270,47,134,136,180,180,True


In [9]:
threshold = 0.8
data['prediction'] = data['similarity'] > threshold
data.head()

,id,leftFileId,leftFilePath,rightFileId,rightFilePath,similarity,totalOverlap,longestFragment,leftCovered,rightCovered,problem1,problem2,ground_truth,prediction
0,24977,99,dataset/python_minifier/problem_054.py,102,dataset/python_minifier/problem_054_minified.py,0.767092,662,50,334,328,054,054,True,False
1,35534,234,dataset/python_minifier/problem_551.py,237,dataset/python_minifier/problem_551_minified.py,0.878906,450,112,225,225,551,551,True,True
2,32861,175,dataset/python_minifier/problem_101.py,172,dataset/python_minifier/problem_101_minified.py,0.623404,293,82,146,147,101,101,True,False
3,31734,154,dataset/python_minifier/problem_089.py,160,dataset/python_minifier/problem_089_minified.py,0.869565,280,79,140,140,089,089,True,True
4,35156,217,dataset/python_minifier/problem_180.py,218,dataset/python_minifier/problem_180_minified.py,0.720000,270,47,134,136,180,180,True,False


In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score


precision = precision_score(data['ground_truth'], data['prediction'])
recall = recall_score(data['ground_truth'], data['prediction'])
f1 = f1_score(data['ground_truth'], data['prediction'])

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"f1-score: {f1:.2f}")
print(f"({precision:.2f}, {recall:.2f}, {f1:.2f})")

Precision: 1.00
Recall: 0.10
f1-score: 0.18
(1.00, 0.10, 0.18)
